# SAMap fibroblast-only analysis

This notebook runs SAMap using fibroblasts only.

The public workflow includes:
- species shown separately and together on the integrated SAMap UMAP;
- checking exact homolog/paralog feature names in each source species;
- homolog/paralog expression in separate panels using original normalized expression;
- homolog/paralog expression in one combined UMAP using a common 0–1 relative-expression scale;
- global Leiden clustering on the integrated fibroblast graph;
- original Seurat clusters projected onto the SAMap UMAP.

Unpublished marker identities and biological conclusions are not embedded.


## Expected inputs and outputs

**Inputs**
- one normalized fibroblast-only `.h5ad` file per species in `outputs/samap/inputs/fibroblast/`;
- pairwise SAMap BLAST maps in `maps/`;
- exported Seurat metadata when original Seurat clusters are projected back onto the integrated embedding.

**Outputs**
- integrated fibroblast SAMap AnnData in `outputs/samap/fibroblast/objects/`;
- species, homolog/paralog, Leiden, and Seurat-cluster figures in `outputs/samap/fibroblast/figures/`;
- cluster-composition and marker tables in `outputs/samap/fibroblast/tables/`;
- run parameters and package versions in `outputs/samap/fibroblast/metadata/`.


In [ ]:
from pathlib import Path

import pandas as pd
import scanpy as sc
from threadpoolctl import threadpool_limits

from samap import SAMAP
from samap.sam import SAM

from python.samap.samap_analysis_utils import (
    load_samap_config,
    species_files_from_config,
    result_directories_from_config,
    validate_h5ad_inputs,
    validate_blast_maps,
    validate_integrated_metadata,
    save_run_metadata,
    SPECIES_ORDER,
    SPECIES_LABELS,
    plot_samap_species,
    plot_samap_species_separate,
    check_gene_presence_across_species,
    check_gene_map_presence,
    check_gene_map_in_sam_species,
    plot_gene_across_sam_species_panels,
    plot_gene_combined_relative_expression,
    run_global_leiden,
    plot_global_leiden_by_species,
    species_composition_by_cluster,
    rank_global_leiden_markers_by_species,
    add_seurat_clusters_from_metadata,
    plot_seurat_clusters_by_species,
)


## Function reference used in this notebook

| Function | What it does |
|---|---|
| `plot_samap_species()` | Shows all species together on the integrated fibroblast SAMap UMAP. |
| `plot_samap_species_separate()` | Shows each species separately while retaining the same integrated SAMap coordinates. |
| `check_gene_presence_across_species()` | Searches the original species AnnData objects for a requested gene and reports exact stored feature names. |
| `check_gene_map_presence()` | Verifies a manually curated homolog/paralog mapping in the original source AnnData objects. |
| `check_gene_map_in_sam_species()` | Verifies the requested homolog/paralog features directly in the species-specific SAM objects used for SAMap. |
| `plot_gene_across_sam_species_panels()` | Plots homolog/paralog expression in separate panels using expression from each species-specific SAM object and coordinates from the integrated SAMap embedding. |
| `plot_gene_combined_relative_expression()` | Shows homolog/paralog expression on one integrated UMAP after within-species 0–1 scaling using the original normalized species AnnData objects. |
| `run_global_leiden()` | Performs one Leiden clustering on the integrated fibroblast SAMap graph. |
| `plot_global_leiden_by_species()` | Displays the global Leiden clusters separately for each species on the shared integrated coordinates. |
| `species_composition_by_cluster()` | Calculates how many cells from each species occur in each global Leiden cluster and their within-cluster proportions. |
| `rank_global_leiden_markers_by_species()` | Finds marker genes for each global Leiden cluster independently within each species. |
| `add_seurat_clusters_from_metadata()` | Adds the original species-specific Seurat cluster assignments back to the integrated SAMap object. |
| `plot_seurat_clusters_by_species()` | Displays each species' original Seurat clusters on the integrated SAMap fibroblast UMAP. |


In [ ]:
PROJECT_DIR = Path(".")
CONFIG = load_samap_config(
    PROJECT_DIR / "config" / "samap_config.json"
)

FILES = species_files_from_config(
    CONFIG,
    workflow="fibroblast",
    project_dir=PROJECT_DIR,
)

MAP_DIR = PROJECT_DIR / CONFIG["paths"]["maps_dir"]

OUT = result_directories_from_config(
    CONFIG,
    workflow="fibroblast",
    project_dir=PROJECT_DIR,
)

SPECIES_ORDER = list(CONFIG["species"])
SPECIES_LABELS = {
    sid: info["name"]
    for sid, info in CONFIG["species"].items()
}

PARAMS = CONFIG["parameters"]


## Validate required inputs

Checks that all fibroblast `.h5ad` files and pairwise BLAST maps are present before starting the cross-species analysis.


In [ ]:
validate_h5ad_inputs(FILES)

validate_blast_maps(
    MAP_DIR,
    SPECIES_ORDER,
)


## Load original normalized fibroblast AnnData objects

Loads the normalized fibroblast-only `.h5ad` objects for each species. These source objects are retained for the combined relative-expression visualization.


In [ ]:
species_adatas = {
    sid: sc.read_h5ad(path)
    for sid, path in FILES.items()
}


## Run fibroblast-only SAMap

Runs SAM independently on the fibroblast population from each species and then aligns the species with SAMap to generate a shared fibroblast manifold.


In [ ]:
sams = {}

for species_id, path in FILES.items():
    sam = SAM()
    sam.load_data(str(path))
    sam.preprocess_data()
    sam.run()
    sams[species_id] = sam

sm_fib = SAMAP(
    sams=sams,
    f_maps=str(MAP_DIR),
)

with threadpool_limits(limits=PARAMS["thread_limit"]):
    sm_fib.run(pairwise=True)

adata = sm_fib.samap.adata
adata.write_h5ad(OUT["objects"] / "integrated.h5ad")


In [ ]:
validate_integrated_metadata(
    adata,
    required_columns={"species"},
)


## Plot all species together on the integrated fibroblast UMAP

Plots all fibroblast cells together on the integrated SAMap UMAP and labels them by species to show how the species occupy the shared fibroblast space.


In [ ]:
plot_samap_species(
    adata,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    save_path=OUT["figures"] / "fibroblast_species_combined.png",
)


## Plot each species separately on the integrated fibroblast UMAP

Shows one species per panel while keeping the same integrated coordinates, making species-specific contributions to the shared fibroblast structure easier to see.


In [ ]:
plot_samap_species_separate(
    adata,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    save_path=OUT["figures"] / "fibroblast_species_separate.png",
)


## Check exact homolog/paralog feature names

Checks which requested homologs or paralogs are actually present and reports their exact feature names before plotting. The source-AnnData and SAM-object checks are kept separate because the two objects can store identifiers differently.


In [ ]:
gene_of_interest = "GENE_OF_INTEREST"

check_gene_presence_across_species(
    species_adatas,
    anchor_gene=gene_of_interest,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    paralog_species=("ze",),
)


In [ ]:
# Example explicit homolog/paralog mapping.
# Replace with the homologs/paralogs you want to inspect.
#
# gene_map = {
#     "hu": ["HUMAN_GENE"],
#     "mo": ["Mouse_gene"],
#     "pi": ["PIG_GENE"],
#     "ch": ["CHICKEN_GENE"],
#     "ze": ["zebrafish_gene_a", "zebrafish_gene_b"],
#     "ti": ["tilapia_gene"],
# }
#
# check_gene_map_presence(
#     species_adatas,
#     gene_map=gene_map,
#     species_order=SPECIES_ORDER,
#     species_labels=SPECIES_LABELS,
# )


## Homolog/paralog expression — separate panels

Uses coordinates from the integrated fibroblast SAMap embedding but takes expression directly from the species-specific SAM objects used in the alignment. Each individual gene/paralog panel is independently scaled to its own 99th-percentile expression value.


In [ ]:
# Verify the curated homolog/paralog names directly in the SAM species objects.
# Define `gene_map` in the section above.
#
# check_gene_map_in_sam_species(
#     sams,
#     gene_map=gene_map,
#     species_order=SPECIES_ORDER,
#     species_labels=SPECIES_LABELS,
# )

# Plot expression from the species-specific SAM objects.
# Coordinates still come from the integrated SAMap UMAP.
#
# plot_gene_across_sam_species_panels(
#     integrated_adata=adata,
#     sams=sams,
#     gene_map=gene_map,
#     species_order=SPECIES_ORDER,
#     species_labels=SPECIES_LABELS,
#     percentile=PARAMS["expression_percentile"],
#     save_path=OUT["figures"] / "homolog_expression_separate.png",
# )


## Homolog/paralog expression — combined UMAP with 0–1 scaling

Uses normalized expression from the original species AnnData objects, combines multiple requested paralogs within a species by taking the per-cell maximum, and rescales each species independently so its 99th percentile equals 1 before plotting all species together.


In [ ]:
plot_gene_combined_relative_expression(
    integrated_adata=adata,
    species_adatas=species_adatas,
    anchor_gene=gene_of_interest,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    paralog_species=("ze",),
    percentile=PARAMS["expression_percentile"],
    save_path=OUT["figures"] / "homolog_expression_combined_0_1.png",
)

# Explicit mapping version:
# plot_gene_combined_relative_expression(
#     integrated_adata=adata,
#     species_adatas=species_adatas,
#     gene_map=gene_map,
#     species_order=SPECIES_ORDER,
#     species_labels=SPECIES_LABELS,
#     percentile=PARAMS["expression_percentile"],
#     save_path=OUT["figures"] / "homolog_expression_combined_0_1.png",
# )


## Global Leiden clustering on the integrated fibroblast graph

Runs a single Leiden clustering on the integrated fibroblast neighbor graph to define cross-species groups based on the shared SAMap structure.


In [ ]:
adata = run_global_leiden(
    adata,
    resolution=PARAMS["global_leiden_resolution"],
    output_col="global_leiden",
)

plot_global_leiden_by_species(
    adata,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    save_path=OUT["figures"] / "global_leiden_by_species.png",
)


## Species composition of global Leiden clusters

Counts and summarizes how much each species contributes to every global Leiden cluster, which helps distinguish broadly shared clusters from species-skewed clusters.


In [ ]:
cluster_counts, species_composition, species_enrichment = (
    species_composition_by_cluster(
        adata,
        cluster_col="global_leiden",
        species_col="species",
    )
)

cluster_counts.to_csv(OUT["tables"] / "global_leiden_species_counts.csv")
species_composition.to_csv(OUT["tables"] / "global_leiden_species_composition.csv")
species_enrichment.to_csv(OUT["tables"] / "global_leiden_species_enrichment.csv")


## Rank markers within global Leiden clusters separately by species

Performs marker ranking independently within each species for the shared global Leiden labels. This avoids treating species differences themselves as cluster markers and allows later comparison of homologous marker programs.


In [ ]:
marker_tables = rank_global_leiden_markers_by_species(
    adata,
    species_ids=SPECIES_ORDER,
    cluster_col="global_leiden",
)

marker_dir = OUT["tables"] / "global_leiden_markers"
marker_dir.mkdir(parents=True, exist_ok=True)

for sid, table in marker_tables.items():
    table.to_csv(
        marker_dir / f"{sid}_global_leiden_markers.csv",
        index=False,
    )


## Project original Seurat clusters onto the fibroblast SAMap UMAP

Adds the original within-species Seurat cluster labels to the integrated SAMap object and plots them on the shared coordinates to compare species-specific clustering with the integrated cross-species structure.


In [ ]:
metadata_files = {
    sid: PROJECT_DIR / "outputs" / "samap_exports" / "fibroblast" / name / "metadata.csv"
    for sid, name in {
        "hu": "human",
        "mo": "mouse",
        "pi": "pig",
        "ch": "chicken",
        "ze": "zebrafish",
        "ti": "tilapia",
    }.items()
}

adata = add_seurat_clusters_from_metadata(
    adata,
    metadata_files=metadata_files,
)

plot_seurat_clusters_by_species(
    adata,
    species_order=SPECIES_ORDER,
    species_labels=SPECIES_LABELS,
    save_path=OUT["figures"] / "seurat_clusters_on_samap_umap.png",
)


## Save run metadata

Saves the exact parameters and installed package versions used for this fibroblast-only SAMap run.


In [ ]:
save_run_metadata(
    OUT["metadata"] / "run_metadata.json",
    parameters=PARAMS,
    extra={
        "workflow": "fibroblast",
        "species": SPECIES_ORDER,
        "input_files": {
            sid: str(path)
            for sid, path in FILES.items()
        },
    },
)
